In [0]:
import numpy as np
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [0]:
df = spark.read.table("workspace.default.capstone_trade_model_data").toPandas()

In [0]:
df = df.dropna()

In [0]:
drop_columns = ["strategy", "entry_date", "exit_date", "pnl_price", "profitable", "Close", "Low", "High", "Date"]

X_columns = [col for col in df.columns if col not in drop_columns]

X = df[X_columns]
y = df["profitable"]

In [0]:
strategies = df["strategy"].dropna().unique()

In [0]:
summary_results = []
all_cv_results = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for strat in strategies:
    print(f"\nRunning strategy: {strat}")
    
    df_strat = df[df["strategy"] == strat].copy()

    # Binary target: 1 = profitable, 0 = not profitable
    y_strat = df_strat["profitable"]

    # Features
    X_strat = df_strat[X_columns].copy()
    X_strat = X_strat.select_dtypes(include=["number"])
    X_strat = X_strat.replace([np.inf, -np.inf], np.nan)

    print("Features used:", X_strat.columns.tolist())

    # Safety checks
    if len(df_strat) < 20:
        print(f"Skipping {strat}: too few rows")
        continue

    if X_strat.shape[1] == 0:
        print(f"Skipping {strat}: no features")
        continue

    if y_strat.nunique() < 2:
        print(f"Skipping {strat}: only one class in target")
        continue

    if y_strat.value_counts().min() < 2:
        print(f"Skipping {strat}: one class has fewer than 2 rows")
        continue

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_strat,
        y_strat,
        test_size=0.20,
        random_state=42,
        stratify=y_strat
    )

    # Make sure 5-fold CV is possible
    if y_train.value_counts().min() < 5:
        print(f"Skipping {strat}: not enough training samples per class for 5-fold CV")
        continue

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1
        ))
    ])

    param_grid = {
        "model__hidden_layer_sizes": [(32,), (64,), (32, 16)],
        "model__activation": ["relu", "tanh"],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.001, 0.01]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        error_score="raise",
        return_train_score=True
    )

    with mlflow.start_run(run_name=f"mlp_clf_{strat}"):
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        print("Best params:", grid.best_params_)
        print("Best CV F1:", round(grid.best_score_, 4))
        print("Test Accuracy:", round(acc, 4))
        print("Test F1:", round(f1, 4))
        print("Test AUC:", round(auc, 4))
        print(classification_report(y_test, y_pred))

        # View probability output
        results = X_test.copy()
        results["actual"] = y_test.values
        results["predicted"] = y_pred
        results["prob_profitable"] = y_prob
        results["prob_profitable_pct"] = y_prob * 100

        print(results[["actual", "predicted", "prob_profitable_pct"]].head())

        # Full grid search results
        cv_results_df = pd.DataFrame(grid.cv_results_).copy()
        cv_results_df["strategy"] = strat

        print(f"\nGrid search results for {strat}:")
        print(
            cv_results_df[
                [
                    "rank_test_score",
                    "mean_test_score",
                    "std_test_score",
                    "mean_train_score",
                    "std_train_score",
                    "mean_fit_time",
                    "params"
                ]
            ].sort_values("rank_test_score")
        )

        all_cv_results.append(cv_results_df)

        summary_results.append({
            "strategy": strat,
            "best_params": str(grid.best_params_),
            "best_cv_f1": grid.best_score_,
            "test_accuracy": acc,
            "test_f1": f1,
            "test_auc": auc
        })

        # MLflow logging
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("strategy", strat)
        mlflow.log_param("test_size", 0.20)

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_auc", auc)
        mlflow.log_metric("best_cv_score", grid.best_score_)

        mlflow.sklearn.log_model(best_model, name=f"mlp_classifier_{strat}")

if len(summary_results) > 0 and len(all_cv_results) > 0:
    summary_df = pd.DataFrame(summary_results)
    all_cv_results_df = pd.concat(all_cv_results, ignore_index=True)

    print("\nOverall strategy summary:")
    print(summary_df.sort_values("test_f1", ascending=False))

    # Save as CSV
    summary_df.to_csv("mlp_strategy_summary.csv", index=False)

    all_cv_results_df["params"] = all_cv_results_df["params"].astype(str)
    all_cv_results_df.to_csv("mlp_all_gridsearch_results.csv", index=False)

    # Save as Delta tables
    spark.createDataFrame(summary_df).write.mode("overwrite").format("delta").saveAsTable(
        "workspace.default.mlp_strategy_summary"
    )

    spark.createDataFrame(all_cv_results_df).write.mode("overwrite").format("delta").saveAsTable(
        "workspace.default.mlp_all_gridsearch_results"
    )
else:
    print("No strategies produced valid results.")

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

experiment = mlflow.get_experiment_by_name("/Users/schylar.srey@uhsp.edu/StockMarket_CapstoneProject/CapstoneProject/Milestone 4/Machine Learning Models/Neural Network 2026-03-22 16_14_27")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

for _, row in runs.iterrows():
    run_id = row["run_id"]
    strat = row["params.strategy"]

    print(f"\nStrategy: {strat}")

    model_uri = f"runs:/{run_id}/mlp_classifier_{strat}"
    model = mlflow.sklearn.load_model(model_uri)

    # Recreate test data 
    df_strat = df[df["strategy"] == strat]

    X = df_strat[X_columns].select_dtypes(include=["number"]).replace([np.inf, -np.inf], np.nan)
    y = df_strat["profitable"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n", cm)